## Regulatory Compliance Pipeline

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (classification_report, confusion_matrix, 
                             average_precision_score)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import joblib

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Load data
df = pd.read_csv("dataset/aiml_df.csv")

In [12]:
df["risk_level"].value_counts().sort_index()

risk_level
1    12
2    64
3    46
4     8
Name: count, dtype: int64

In [13]:
# Label (high risk = 3 or 4)
df = df[df['risk_level'].notnull()].copy()
y = (df['risk_level'] >= 3).astype(int)

# Text features extraction with tf-idf vectorizer
text = df['indications_for_use'].fillna('') if 'indications_for_use' in df else pd.Series(['']*len(df))
tfidf = TfidfVectorizer(
    ngram_range=(1,2), min_df=2, max_df=0.95,
    max_features=2000, stop_words='english'
)
X_text = tfidf.fit_transform(text)

# encoding categorical variables, if not categorial, put to csr_matrix
cat_cols = [c for c in ['modality','body_area','regulation_medical_specialty'] if c in df.columns]
if cat_cols:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
    X_cat = ohe.fit_transform(df[cat_cols].fillna('unknown'))
    cat_names = ohe.get_feature_names_out(cat_cols).tolist()
else:
    X_cat = csr_matrix((len(df), 0))
    cat_names = []


num = df[['sample_size','num_sites','is_prospective']].apply(pd.to_numeric, errors='coerce').fillna(0.0).values
from scipy.sparse import csr_matrix
X = hstack([X_text, X_cat, csr_matrix(num)]).tocsr()

feature_names = tfidf.get_feature_names_out().tolist() + cat_names

# Single split (bigger test for balance)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.4, stratify=y, random_state=RANDOM_STATE
)

print("Train positives:", int(y_tr.sum()), "of", len(y_tr))
print("Test  positives:", int(y_te.sum()), "of", len(y_te))

# Baseline model (balanced Logistic Regression)
clf = LogisticRegression(class_weight='balanced', solver='liblinear', max_iter=2000, random_state=RANDOM_STATE)
clf.fit(X_tr, y_tr)

# >>> Tiny validation split from TRAIN to tune threshold (keep test untouched)
X_tr2, X_val, y_tr2, y_val = train_test_split(
    X_tr, y_tr, test_size=0.25, stratify=y_tr, random_state=42
)
clf.fit(X_tr2, y_tr2)

# final fit on full train portion
clf.fit(X_tr, y_tr)

joblib.dump(clf,"saved_models/clf.joblib") # saving model
clf = joblib.load("saved_models/clf.joblib") # loading model

proba_te = clf.predict_proba(X_te)[:, 1]
pred_te  = (proba_te >= 0.5).astype(int)
print(classification_report(y_te, pred_te, digits=3))
print("Confusion matrix:\n", confusion_matrix(y_te, pred_te))
print("PR-AUC (positive):", round(average_precision_score(y_te, proba_te), 3))


Train positives: 32 of 78
Test  positives: 22 of 52
              precision    recall  f1-score   support

           0      0.812     0.867     0.839        30
           1      0.800     0.727     0.762        22

    accuracy                          0.808        52
   macro avg      0.806     0.797     0.800        52
weighted avg      0.807     0.808     0.806        52

Confusion matrix:
 [[26  4]
 [ 6 16]]
PR-AUC (positive): 0.883


In [14]:
rf = RandomForestClassifier(n_estimators=300, max_depth=6, class_weight='balanced', random_state=RANDOM_STATE)

In [ ]:
rf.fit(X_tr, y_tr)

proba_rf = rf.predict_proba(X_te)[:,1]
pred_rf = (proba_rf >= 0.5).astype(int)
print("\n=== RandomForest (balanced, depth=6) ===")
print(classification_report(y_te, pred_rf, digits=3))
print("Confusion matrix:\n", confusion_matrix(y_te, pred_rf))
print("PR-AUC (positive class):", round(average_precision_score(y_te, proba_rf), 3))


=== RandomForest (balanced, depth=6) ===
              precision    recall  f1-score   support

           0      0.743     0.867     0.800        30
           1      0.765     0.591     0.667        22

    accuracy                          0.750        52
   macro avg      0.754     0.729     0.733        52
weighted avg      0.752     0.750     0.744        52

Confusion matrix:
 [[26  4]
 [ 9 13]]
PR-AUC (positive class): 0.785


In [16]:
from sklearn.calibration import CalibratedClassifierCV
cal_rf = CalibratedClassifierCV(
    RandomForestClassifier(n_estimators=300, max_depth=6, class_weight='balanced', random_state=RANDOM_STATE),
    cv=3
)

In [18]:
cal_rf.fit(X_tr, y_tr)
joblib.dump(cal_rf,"saved_models/rf.joblib") # saving model
cal_rf = joblib.load("saved_models/rf.joblib") # loading model

proba_cal_rf = cal_rf.predict_proba(X_te)[:,1]
pred_cal_rf = (proba_cal_rf >= 0.5).astype(int)
print("\n=== Calibrated RandomForest (balanced, depth=6) ===")
print(classification_report(y_te, pred_cal_rf, digits=3))
print("Confusion matrix:\n", confusion_matrix(y_te, pred_cal_rf))
print("PR-AUC (positive class):", round(average_precision_score(y_te, proba_cal_rf), 3))


=== Calibrated RandomForest (balanced, depth=6) ===
              precision    recall  f1-score   support

           0      0.707     0.967     0.817        30
           1      0.909     0.455     0.606        22

    accuracy                          0.750        52
   macro avg      0.808     0.711     0.711        52
weighted avg      0.793     0.750     0.728        52

Confusion matrix:
 [[29  1]
 [12 10]]
PR-AUC (positive class): 0.788
